# Config B — Tinh chỉnh Random Forest

## Mục tiêu

Tìm bộ tham số Random Forest có MAE trung bình thấp nhất qua 3 fold theo thời gian trên tập train. Giữ 27 đặc trưng thời gian, ngày lễ và thời tiết như thí nghiệm ngày 13/11.

Mốc đối chiếu ngày 13/11 là MAE CV **281,28** và MAE validation **240,10 xe/giờ**, với 400 cây, độ sâu 12, tối thiểu 5 mẫu mỗi lá và `max_features=1.0`.

- Lượt 1 thử độ sâu và số mẫu mỗi lá, giữ nguyên số cây và mặc định dùng toàn bộ đặc trưng ở mỗi nút.
- Lượt 2 là tùy chọn, thử tỉ lệ đặc trưng ở mỗi nút cho hai bộ đứng đầu lượt 1.
- Validation dùng bộ được chọn theo CV. Bật `RUN_VALIDATION` sau khi xem bảng CV.

## Params

In [ ]:
EXPERIMENT_NAME = "16-11-2026_ConfigB_RandomForest_Phuc"
CONFIG_NAME = "B"
MODEL_NAME = "Random Forest"
TARGET = "traffic_volume"
RANDOM_STATE = 42
CV_SPLITS = 3
RUN_MAX_FEATURES_CV = False
RUN_VALIDATION = False

DATA_PARAMS = {"mask_stuck_temp": False, "complete_holidays": False}

REFERENCE_PARAMS = {
    "n_estimators": 400,
    "max_depth": 12,
    "min_samples_leaf": 5,
    "max_features": 1.0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
}

PARAM_GRIDS = {
    MODEL_NAME: {
        "max_depth": [8, 10, 12, 14],
        "min_samples_leaf": [3, 5, 8, 12],
    },
}
MAX_FEATURES_TO_TRY = [0.6, 0.8]
REFERENCE_VALIDATION_MAE = 240.09808540736057

## Import và tìm thư mục dự án

In [ ]:
import json
import sys
import warnings
from pathlib import Path
from datetime import datetime, timezone, timedelta
from time import perf_counter

import numpy as np
import pandas as pd
from IPython.display import display


def find_project_root():
    current = Path.cwd().resolve()

    for folder in (current, *current.parents):
        for candidate in (
            folder,
            folder / "metro_interstate_traffic_volume",
        ):
            if (
                (candidate / "src" / "models.py").is_file()
                and (candidate / "src" / "features.py").is_file()
            ):
                return candidate

    raise FileNotFoundError(
        "Không tìm thấy dự án. Hãy kiểm tra thư mục làm việc của kernel."
    )


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import features
from src.data import load_splits, load_hourly_splits
from src.evaluation import clip_prediction, regression_metrics

DATA_PATH = PROJECT_ROOT / "data" / "Metro_Interstate_Traffic_Volume.csv"

RUN_ID = datetime.now(
    timezone(timedelta(hours=7))
).strftime("%Y%m%d_%H%M%S_%f")

OUTPUT_DIR = (
    PROJECT_ROOT
    / "experiments"
    / EXPERIMENT_NAME
    / "outputs"
    / RUN_ID
)

print("Project:", PROJECT_ROOT)
print("Data:", DATA_PATH)
print("Output:", OUTPUT_DIR)

assert DATA_PATH.is_file(), f"Không tìm thấy dữ liệu: {DATA_PATH}"
REFERENCE_DIR = PROJECT_ROOT / "experiments" / "13-11-2026_ConfigB_4Model_Phuc"
# Nạp helper bằng đường dẫn cụ thể khi kernel được mở từ thư mục khác.
import importlib.util
spec = importlib.util.spec_from_file_location(
    "config_b_grid_cv", REFERENCE_DIR / "config_b_grid_cv.py"
)
grid_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(grid_module)
make_config_b_pipeline = grid_module.make_config_b_pipeline
run_grid_cv = grid_module.run_grid_cv

## Đọc dữ liệu train và kiểm tra Config B

In [ ]:
FEATURE_COLUMNS = list(features.CONFIG_B)
raw_train = load_hourly_splits(path=DATA_PATH, **DATA_PARAMS)["train"]

assert len(FEATURE_COLUMNS) == 27
assert TARGET not in FEATURE_COLUMNS
assert set(FEATURE_COLUMNS).isdisjoint(features.HISTORY_FEATURES)
assert raw_train["date_time"].is_monotonic_increasing
assert not raw_train["date_time"].duplicated().any()

display(pd.DataFrame([{
    "split": "train",
    "rows": len(raw_train),
    "start": raw_train["date_time"].min(),
    "end": raw_train["date_time"].max(),
    "n_features": len(FEATURE_COLUMNS),
}]))
print("Features:", FEATURE_COLUMNS)

## Lượt 1 · Grid CV theo thời gian

Gọi lại helper của thí nghiệm ngày 13/11 để xử lý thời tiết, tạo đặc trưng và tính MAE giống lần trước. Lưới gồm 16 tổ hợp và 48 lần fit, kể cả bộ đối chiếu. Nếu bộ dẫn đầu nằm ở biên lưới, có thể mở rộng đúng phía đó trong `PARAM_GRIDS` trước khi thử lượt 2.

In [ ]:
cv_depth_leaf = run_grid_cv(
    raw_train, {MODEL_NAME: REFERENCE_PARAMS}, PARAM_GRIDS, n_splits=CV_SPLITS,
).assign(stage="depth_leaf")

if not any(params == REFERENCE_PARAMS for params in cv_depth_leaf["params"]):
    reference_cv = run_grid_cv(
        raw_train, {MODEL_NAME: REFERENCE_PARAMS}, {MODEL_NAME: {}}, n_splits=CV_SPLITS,
    ).assign(stage="reference")
    cv_depth_leaf = pd.concat([cv_depth_leaf, reference_cv], ignore_index=True)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cv_depth_leaf.to_csv(OUTPUT_DIR / "grid_cv_depth_leaf.csv", index=False)

display(cv_depth_leaf.sort_values("mean_MAE", kind="stable").reset_index(drop=True))

## Lượt 2 · Thử thêm max_features

Khi cần, đặt `RUN_MAX_FEATURES_CV=True` ở cell Params, chạy lại cell Params và cell bên dưới. Hai bộ dẫn đầu lượt 1 được thử với `max_features=0.6` và `0.8`. Điểm của `1.0` lấy từ lượt 1, nên lượt này có thêm 12 lần fit.

In [ ]:
cv_parts = [cv_depth_leaf]
if RUN_MAX_FEATURES_CV:
    top_two = cv_depth_leaf.nsmallest(2, "mean_MAE")
    for params in top_two["params"]:
        extra_results = run_grid_cv(
            raw_train,
            {MODEL_NAME: params},
            {MODEL_NAME: {"max_features": MAX_FEATURES_TO_TRY}},
            n_splits=CV_SPLITS,
        )
        cv_parts.append(extra_results.assign(stage="max_features"))

cv_results = (
    pd.concat(cv_parts, ignore_index=True)
    .sort_values("mean_MAE", kind="stable")
    .reset_index(drop=True)
)
reference = next(
    row for row in cv_depth_leaf.itertuples() if row.params == REFERENCE_PARAMS
)
best = cv_results.iloc[0]
best_params = dict(best["params"])

mae_columns = [f"fold_{fold}_MAE" for fold in range(1, CV_SPLITS + 1)]
mae_columns += ["mean_MAE", "std_MAE"]
comparison = pd.DataFrame(
    [cv_results.iloc[0][mae_columns], pd.Series({key: getattr(reference, key) for key in mae_columns})],
    index=["Bộ tốt nhất", "Bộ đối chiếu"],
)
display(comparison.round(4))
display(cv_results)
print("Tham số tốt nhất:", best_params)

cv_export = cv_results.copy()
cv_export["params"] = cv_export["params"].map(
    lambda params: json.dumps(params, ensure_ascii=False)
)
cv_export.to_csv(OUTPUT_DIR / "grid_cv_results.csv", index=False)
(OUTPUT_DIR / "grid_cv_config.json").write_text(
    json.dumps({
        "n_splits": CV_SPLITS,
        "data_params": DATA_PARAMS,
        "reference_params": REFERENCE_PARAMS,
        "param_grids": PARAM_GRIDS,
        "max_features_cv_enabled": RUN_MAX_FEATURES_CV,
        "max_features_to_try": MAX_FEATURES_TO_TRY,
    }, indent=2),
    encoding="utf-8",
)
(OUTPUT_DIR / "best_params.json").write_text(
    json.dumps(best_params, indent=2), encoding="utf-8",
)

## Huấn luyện bộ đã chọn và đánh giá validation

Sau khi xem MAE từng fold, đặt `RUN_VALIDATION=True` trong cell Params, chạy lại cell Params rồi chạy hai cell cuối. So sánh với mốc **240,10 xe/giờ**. Tập test chưa dùng trong thí nghiệm này.

In [ ]:
if RUN_VALIDATION:
    splits = load_splits(path=DATA_PATH, **DATA_PARAMS)
    featured = {
        name: features.add_weather_features(features.add_time_features(splits[name]))
        for name in ("train", "validation")
    }
    train = featured["train"]
    validation = featured["validation"]
    assert train["date_time"].max() < validation["date_time"].min()

    pipeline = make_config_b_pipeline(MODEL_NAME, best_params)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        start = perf_counter()
        pipeline.fit(train[FEATURE_COLUMNS], train[TARGET])
        fit_seconds = perf_counter() - start

        start = perf_counter()
        raw_prediction = pipeline.predict(validation[FEATURE_COLUMNS])
        predict_seconds = perf_counter() - start

    assert np.isfinite(raw_prediction).all()
    prediction = clip_prediction(raw_prediction)
    metrics = regression_metrics(validation[TARGET], prediction)
    results = pd.DataFrame([{
        "config": CONFIG_NAME,
        "model": MODEL_NAME,
        "split": "validation",
        "n_features": len(FEATURE_COLUMNS),
        "n_train": len(train),
        "n_validation": len(validation),
        "fit_seconds": fit_seconds,
        "predict_seconds": predict_seconds,
        **metrics,
    }])
    predictions = validation[["date_time", TARGET]].copy()
    predictions[f"{MODEL_NAME}__raw"] = raw_prediction
    predictions[MODEL_NAME] = prediction
    warning_table = pd.DataFrame(
        [{"model": MODEL_NAME, "category": w.category.__name__, "message": str(w.message)} for w in caught],
        columns=["model", "category", "message"],
    )

    display(results.round(4))
    if not warning_table.empty:
        display(warning_table)
    print(f"Thay đổi MAE so với ngày 13/11: {metrics['MAE'] - REFERENCE_VALIDATION_MAE:+.2f} xe/giờ")
    results.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)
    predictions.to_csv(OUTPUT_DIR / "validation_predictions.csv", index=False)
    warning_table.to_csv(OUTPUT_DIR / "validation_warnings.csv", index=False)
else:
    print("Validation đang tắt. Bật RUN_VALIDATION sau khi xem kết quả CV.")

In [ ]:
if RUN_VALIDATION:
    import matplotlib.dates as mdates
    import matplotlib.pyplot as plt

    hours_to_plot = 7 * 24
    plot_index = pd.date_range(
        start=predictions["date_time"].min(), periods=hours_to_plot, freq="h",
    )
    plot_data = predictions.set_index("date_time").reindex(plot_index)

    fig, ax = plt.subplots(figsize=(14, 4), layout="constrained")
    ax.plot(plot_index, plot_data[TARGET], color="#202020", label="Thực tế")
    ax.plot(
        plot_index, plot_data[MODEL_NAME], color="tab:orange",
        linestyle="--", label="Dự đoán",
    )
    ax.set_title("Random Forest · Config B · 168 giờ đầu validation")
    ax.set_xlabel("Ngày và giờ")
    ax.set_ylabel("Xe/giờ")
    ax.xaxis.set_major_locator(mdates.HourLocator(byhour=[0, 12]))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d/%m\n%H:%M"))
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right")

    image_dir = OUTPUT_DIR / "img"
    image_dir.mkdir(exist_ok=True)
    fig.savefig(image_dir / "config_b_random_forest_168h.png", dpi=300, facecolor="white")
    plt.show()

## Nhận xét sau khi chạy

- Ghi bộ tham số có MAE CV thấp nhất và mức thay đổi ở cả ba fold.
- Ghi MAE, RMSE và R² trên validation. So sánh MAE với ngày 13/11.
- Xem biểu đồ 168 giờ đầu và ghi các đoạn sai lệch lớn vào phần nhận xét.